### ASTRADB VectorStore
Go from app idea to production with the AI Platform with Astra DB, the ultra-low latency database made for AI and Langflow, the low-code RAG IDE
https://www.datastax.com/

In [19]:
### ASTRADB VectorStore

In [20]:
# Install LangChain ecosystem
"""
!pip install -q \
langchain \
langchain-core \
langchain-groq \
langchain-google-genai \
langchain-astradb
"""

'\n!pip install -q langchain langchain-core langchain-groq langchain-google-genai langchain-astradb\n'

In [21]:
from google.colab import userdata
import os

In [22]:
### Config
ASTRA_DB_API_ENDPOINT="https://27badaf2-1d4e-4d74-91c9-fcc7b8807e49-us-east-2.apps.astra.datastax.com"
ASTRA_DB_APPLICATION_TOKEN=userdata.get('astra_db_token')

In [23]:
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [24]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
)

In [25]:
embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x790cd16a1370>, model='gemini-embedding-2', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

In [26]:
from langchain_astradb import AstraDBVectorStore
vector_store = AstraDBVectorStore(
    embedding=embeddings,
    api_endpoint=ASTRA_DB_API_ENDPOINT,
    collection_name="astra_vector_langchain",
    token=ASTRA_DB_APPLICATION_TOKEN,
    namespace=None,
)
vector_store

In [27]:
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]
documents

[Document(metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.'),
 Document(metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(metadata={'source': 'website'}, page_content='Is the new iPhone worth the price? Read this review to find out.'),
 Document(metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.'),
 Document(metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic application

In [28]:
vector_store.add_documents(documents=documents)

['feb0b7d5472340798de5b385f6f01476',
 'b3457254400041c29787f354d54b03eb',
 '9673f455e5584efe8ccc65c9a8cc56b3',
 '4d0b33c6b834496c9eadc18b21a8d430',
 'aa8120db12c943dda670c186cccf2336',
 '7319434c43f7409088669035ec190d72',
 '604e8aebb06d469eb685b8ebdf0fb2a9',
 '37cc3926657f407b92b61c20717d343f',
 'a94ae181316743faa36b175d45d64090',
 '8c97daa03a3846448578a72c3b3399cd']

In [29]:
### Search from Vector Store DB

vector_store.similarity_search("What is the weather")

[Document(id='cc4b9d60f2904b8aaf585467ca0a44a4', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='001e153a02be4bb0a17e2fa788b14c2b', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='b3457254400041c29787f354d54b03eb', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='8c97daa03a3846448578a72c3b3399cd', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [30]:
results = vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for res in results:
    print(f'* "{res.page_content}", metadata={res.metadata}')

* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}


In [31]:
results = vector_store.similarity_search_with_score(
    "LangChain provides abstractions to make working with LLMs easy",
    k=3,
    filter={"source": "tweet"},
)
for res, score in results:
    print(f'* [SIM={score:.2f}] "{res.page_content}", metadata={res.metadata}')

* [SIM=0.89] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* [SIM=0.89] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}
* [SIM=0.89] "Building an exciting new project with LangChain - come check it out!", metadata={'source': 'tweet'}


In [32]:
### Retriever
retriever = vector_store.as_retriever(
  search_type="similarity_score_threshold",
    search_kwargs={"k": 1, "score_threshold": 0.5},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='376f6fac79e0429aafe0b6b8467879a4', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]

In [33]:
### Retriever
retriever = vector_store.as_retriever(
  search_type="mmr",
    search_kwargs={"k": 1},
)
retriever.invoke("Stealing from the bank is a crime", filter={"source": "news"})

[Document(id='376f6fac79e0429aafe0b6b8467879a4', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]